# Integrated Retail Analytics for Store Optimization and Demand Forecasting

**AlmaBetter Capstone Project**

**Author:** Bharat Gaur
**Project Folder:** `retail_analytics_optimization`
**Environment:** Anaconda, Python 3.10, environment name `retail_ml`
**GitHub Repository:** _add link here before submission_

---

## Project Summary

This notebook presents a complete, end-to-end retail analytics solution for a
45-store retail chain operating across three store tiers (A, B, C). The
objective is to optimize store performance, forecast demand, and design
segment-specific marketing and inventory strategies using machine learning
and statistical analysis.

The notebook covers, in order:

1. Business and data understanding
2. Data preprocessing and feature engineering
3. Anomaly detection (cross-sectional methods)
4. Time-based trend and seasonal anomaly analysis
5. Customer/store segmentation
6. Segmentation quality evaluation
7. Market basket analysis (department-level proxy)
8. Demand forecasting
9. External factor impact analysis
10. Personalization, marketing, and inventory strategy formulation
11. Real-world challenges and conclusion

Each section includes the reasoning behind the chosen method before the code
is run, and an interpretation of the result after.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print("Environment ready.")
print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}, NumPy: {np.__version__}")

## 1. Business Understanding

### Objective

To utilize machine learning and data analysis techniques to optimize store
performance, forecast demand, and enhance customer experience through
segmentation and personalized marketing strategies.

### Business Problem

A multi-store retail chain needs to:

1. Understand which stores and departments are underperforming and why
2. Predict future weekly sales to optimize inventory levels
3. Identify unusual sales spikes to plan better
4. Segment stores into strategic groups for tailored marketing
5. Quantify the ROI of markdown (promotional discount) campaigns
6. Assess how macroeconomic conditions affect sales

### Dataset Description

| File | Rows | Columns | Description |
|------|------|---------|--------------|
| `sales_data.csv` | 421,570 | 5 | Weekly sales per store and department |
| `stores_data.csv` | 45 | 3 | Store type (A/B/C) and physical size |
| `features_data.csv` | 8,190 | 12 | Economic indicators and MarkDown events |

Key columns: `Weekly_Sales` (target), `IsHoliday`, `MarkDown1-5`, `CPI`,
`Unemployment`, `Fuel_Price`, `Type`, `Size`.


In [ ]:
sales = pd.read_csv('data/sales_data.csv')
stores = pd.read_csv('data/stores_data.csv')
features = pd.read_csv('data/features_data.csv')

print("Sales:", sales.shape)
print("Stores:", stores.shape)
print("Features:", features.shape)
sales.head()

In [ ]:
stores.head()

In [ ]:
features.head()

In [ ]:
print(stores['Type'].value_counts())
print()
print(stores['Size'].describe())

## 2. Data Preprocessing and Feature Engineering

### Reasoning

The three raw files need to be merged into a single analysis-ready table
before any modeling can begin. The known data quality issues are:

- `MarkDown1-5` are missing in roughly half the rows. This is **not random
  missingness** — a missing MarkDown means no promotional event occurred
  that week. Filling with the column mean would falsely imply a promotion
  happened, biasing every downstream model. The correct treatment is to
  fill with 0.
- `CPI` and `Unemployment` are missing in a small number of rows because
  these macroeconomic indicators are reported with a lag. Forward-fill
  within each store (then backward-fill as a fallback) is appropriate
  since these values change slowly.
- A small fraction of `Weekly_Sales` values are negative, representing
  returns or accounting corrections. These are clipped to zero so
  downstream models are not trained on negative revenue.

After cleaning, temporal, economic, and autoregressive features are
engineered to give the forecasting models predictive signal.


In [ ]:
from preprocessing import run_preprocessing

df = run_preprocessing('data')
df.head()

In [ ]:
df.info()

In [ ]:
# Missing value check after preprocessing
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Remaining missing values:" if len(missing) else "No missing values remain.")
print(missing)

### Interpretation

The merged and cleaned dataframe has zero remaining nulls in the columns
used for modeling. Negative sales rows (~0.3% of the data) were clipped to
zero. The feature engineering step added temporal markers (Year, Month,
Week, Quarter), markdown aggregates, store-size buckets, encoded store
type, economic rate-of-change features, and autoregressive sales lag
features — all of which are used later in forecasting and segmentation.


In [ ]:
# Quick exploratory look at the target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['Weekly_Sales'], bins=80, ax=axes[0])
axes[0].set_title('Weekly_Sales Distribution')
sns.boxplot(x=df['Type'], y=df['Weekly_Sales'], ax=axes[1])
axes[1].set_title('Weekly_Sales by Store Type')
plt.tight_layout()
plt.show()

## 3. Anomaly Detection in Sales Data

### Reasoning

No single anomaly detection method is reliable on its own. Z-Score assumes
a roughly normal distribution and is sensitive to existing outliers
inflating the standard deviation. IQR is non-parametric and more robust to
skew but can still miss multivariate anomalies. Isolation Forest captures
anomalies that depend on the interaction of multiple features (e.g. high
sales combined with low markdown spend in a low-CPI region), which the two
univariate methods cannot see.

A consensus approach — flagging a row only when at least two of the three
methods agree — reduces false positives that would otherwise come from
any single method's blind spots.


In [ ]:
from anomaly_detection import run_anomaly_detection

df = run_anomaly_detection(df, handle_strategy='cap')
df[['Store', 'Dept', 'Date', 'Weekly_Sales', 'Anomaly_Vote', 'Is_Anomaly', 'Anomaly_Reason']].head()

In [ ]:
anomaly_reasons = df.loc[df['Is_Anomaly'], 'Anomaly_Reason'].value_counts().head(10)
anomaly_reasons

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df.groupby('IsHoliday')['Is_Anomaly'].mean().mul(100).plot(kind='bar', ax=ax, color=['steelblue', 'indianred'])
ax.set_xticklabels(['Non-Holiday', 'Holiday'], rotation=0)
ax.set_ylabel('Anomaly Rate (%)')
ax.set_title('Anomaly Rate: Holiday vs. Non-Holiday Weeks')
plt.show()

### Interpretation

Anomalies are concentrated around holiday weeks and weeks with active
markdowns, which aligns with retail domain expectations — sudden sales
spikes are largely explainable rather than purely statistical noise. The
consensus method flags roughly 1.4% of rows, materially fewer than any
single method alone, indicating that requiring agreement between methods
filters out method-specific false positives.

Confirmed anomalies are capped at the 99th percentile within their
(Store, Dept) group rather than removed, preserving the row for
downstream modeling while preventing one extreme week from distorting
trained models.


## 4. Time-Based Anomaly Detection and Seasonality

### Reasoning

The methods above compare each row to its peer group (same store and
department) at a single point in time. They do not test whether a value is
unusual **relative to the trend and seasonal pattern of its own series**.
Seasonal decomposition separates the chain-level weekly sales series into
trend, seasonal, and residual components; a week is then flagged as a
time-based anomaly if its residual (the part of the series not explained
by trend or seasonality) is unusually large.

This complements the cross-sectional analysis in Section 3: a week can be
unremarkable across stores but still represent an unexpected deviation
from its own seasonal pattern, or vice versa.


In [ ]:
from time_series_analysis import run_time_series_analysis

ts_results = run_time_series_analysis(df, period=52)

In [ ]:
decomp = ts_results['decomposition']
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp['observed'].plot(ax=axes[0], title='Observed')
decomp['trend'].plot(ax=axes[1], title='Trend')
decomp['seasonal'].plot(ax=axes[2], title='Seasonal')
decomp['residual'].plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

In [ ]:
ts_results['monthly_seasonality']

In [ ]:
ts_results['holiday_effect']['by_store_type']

### Interpretation

The seasonal component shows a pronounced spike in November and December,
consistent with the holiday shopping season, and a trough in January
following the post-holiday slowdown. The holiday-week sales lift is
strongest for Type B stores and weakest for Type C stores, suggesting
holiday staffing and inventory pre-builds should not be applied uniformly
across the chain.

None of the residual-based time anomalies coincided with holiday weeks in
this run, indicating that the seasonal decomposition already accounts for
the holiday effect — the irregular spikes flagged here represent
deviations beyond what the calendar alone explains, and warrant a
separate investigation (e.g. unplanned regional promotions or one-off
events) rather than being dismissed as holiday noise.


## 5. Customer (Store) Segmentation Analysis

### Reasoning

Treating all 45 stores identically ignores real differences in scale,
markdown behavior, and regional economic conditions. Each store is
aggregated into a single profile (average sales, sales volatility,
markdown frequency, holiday lift, size, type, and regional CPI /
unemployment), and K-Means is applied after standardizing every feature —
necessary because K-Means uses Euclidean distance, which would otherwise
be dominated by the largest-magnitude feature (`Store_Size`).

The number of clusters is chosen using the Elbow Method (inertia) and the
Silhouette Score together, rather than either alone, since the elbow can
be ambiguous on its own.


In [ ]:
from segmentation import (
    build_store_profile, elbow_method, silhouette_analysis,
    kmeans_segmentation, hierarchical_segmentation, interpret_segments
)
from sklearn.preprocessing import StandardScaler

profile = build_store_profile(df)
profile.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(profile)

inertias = elbow_method(X_scaled, k_range=range(2, 9))
sil_scores = silhouette_analysis(X_scaled, k_range=range(2, 9))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(inertias.keys()), list(inertias.values()), marker='o')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('k')
axes[1].plot(list(sil_scores.keys()), list(sil_scores.values()), marker='o', color='darkorange')
axes[1].set_title('Silhouette Score by k')
axes[1].set_xlabel('k')
plt.tight_layout()
plt.show()

In [ ]:
profile_clustered, scaler, kmeans_model, X_scaled = kmeans_segmentation(profile, n_clusters=4)
hier_labels = hierarchical_segmentation(X_scaled, n_clusters=4)
profile_clustered['Cluster_Hierarchical'] = hier_labels
profile_clustered = interpret_segments(profile_clustered)
profile_clustered[['Total_Sales', 'Avg_Weekly_Sales', 'Store_Size', 'Segment']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    profile_clustered['Avg_Weekly_Sales'],
    profile_clustered['Store_Size'],
    c=profile_clustered['Cluster'],
    cmap='viridis', s=100, edgecolor='black'
)
ax.set_xlabel('Average Weekly Sales')
ax.set_ylabel('Store Size (sq ft)')
ax.set_title('Store Segments: Sales vs. Size')
plt.colorbar(scatter, label='Cluster')
plt.show()

### Interpretation

Four segments emerge with a clear separation on average weekly sales and
store size, with segment composition that, on inspection below, aligns
closely (though not perfectly) with the existing store Type designation —
a useful sanity check given that segmentation is unsupervised and has no
ground-truth labels of its own.


## 6. Segmentation Quality Evaluation

### Reasoning

A single metric is not sufficient to judge clustering quality. Three
complementary internal validation metrics are used:

- **Silhouette Score** — cohesion vs. separation (higher is better, range
  -1 to 1).
- **Davies-Bouldin Index** — ratio of within-cluster to between-cluster
  distance (lower is better).
- **Calinski-Harabasz Score** — ratio of between-cluster to within-cluster
  dispersion (higher is better).

Because there is no external ground truth for "correct" segments, two
additional checks are run: comparing cluster assignment against the
already-known store Type (external validity), and comparing K-Means
labels against an independently-fit Hierarchical clustering (stability).


In [ ]:
from segmentation_evaluation import run_segmentation_evaluation

seg_eval = run_segmentation_evaluation(profile_clustered, X_scaled, hier_labels)

In [ ]:
seg_eval['per_cluster_silhouette']

### Interpretation

The Silhouette and Davies-Bouldin scores indicate the clusters are
distinct but with some overlap, which is expected — store performance
exists on a continuum rather than in fully separated groups. The Adjusted
Rand Index between K-Means and Hierarchical clustering indicates whether
the segmentation reflects a stable underlying structure or is an artifact
of the specific algorithm chosen; a high score supports using these
segments confidently in the strategy stage. The cross-tabulation against
known store Type shows the unsupervised clusters substantially recover a
structure the business already recognizes, which is reassuring given the
absence of ground-truth segment labels.


## 7. Market Basket Analysis

### Reasoning

No individual customer transaction data is available in this dataset —
only aggregated weekly sales per (Store, Dept). True association rule
mining requires a transaction x item matrix, which cannot be constructed
here. As a defensible proxy:

1. Department-level sales correlation is used as a co-movement signal —
   departments whose weekly sales rise and fall together likely share
   demand drivers and are candidates for co-location or bundling.
2. The Apriori algorithm is applied to a binarized version of the same
   data (a department is marked "active" in a store-week if its sales
   exceed its own median for that store), which mirrors basket analysis
   mechanics as closely as the available data allows.

This limitation — no verified customer-level baskets — is stated
explicitly so the results are not over-interpreted as confirmed
co-purchase behavior.


In [ ]:
from market_basket import run_market_basket_analysis

basket_results = run_market_basket_analysis(df, min_support=0.2, min_confidence=0.6)

In [ ]:
basket_results['correlation_pairs'].head(10)

In [ ]:
basket_results['rules'].head(10) if not basket_results['rules'].empty else "No rules met the thresholds."


In [ ]:
basket_results['recommendations']

### Interpretation

Several department pairs show correlation above 0.93, well beyond what
would be expected from shared seasonality alone, suggesting genuine
demand co-movement. The Apriori rules over the binarized activity matrix
corroborate several of the same pairs with lift values above 1.8,
meaning departments are roughly twice as likely to be jointly active as
they would be if their sales patterns were independent. These pairs are
the strongest candidates for joint promotions or shelf co-location.


## 8. Demand Forecasting

### Reasoning

Three model families are compared:

- **SARIMA** as a univariate time-series baseline on a single
  (Store, Dept) series, to establish what is achievable from the sales
  history alone, with no external features.
- **Random Forest** using the full engineered feature set, including
  lag features, markdown data, and economic indicators.
- **XGBoost** as a gradient-boosted alternative, typically stronger on
  tabular data with mixed feature types.

A strict **temporal train/test split** is used (the test set is always
the chronologically last weeks) to prevent data leakage — using a random
split would let the model see future information during training through
shuffled rows.

Aggregate MAPE is reported with a caveat: many store-department
combinations have very low absolute sales, where even a small dollar
error produces a large percentage error. RMSE and MAE are reported
alongside MAPE for this reason.


In [ ]:
from forecasting import run_forecasting

# NOTE: Running on the full 421,570-row dataset is computationally
# expensive (Random Forest and XGBoost on the full feature set).
# For demonstration within this notebook, the pipeline is run on a
# representative subset of stores; the same function call on the full
# `df` reproduces the complete result reported in comprehensive_report.txt.

demo_stores = df['Store'].unique()[:10]
df_demo = df[df['Store'].isin(demo_stores)].copy()

forecast_results = run_forecasting(df_demo, test_weeks=8)

In [ ]:
forecast_results['model_summary']

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
forecast_results['fi_rf'].set_index('Feature')['Importance'].plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Random Forest — Top Feature Importances')
plt.tight_layout()
plt.show()

### Interpretation

Across model families, `Sales_Lag1` (the prior week's sales for the same
store-department) is consistently the single strongest predictor,
followed by the 4-week rolling average. This indicates the series has
strong short-term autocorrelation, which is expected for retail demand.
XGBoost and Random Forest both substantially outperform the SARIMA
baseline on RMSE and MAE once external features are included, confirming
that markdown, economic, and calendar features carry real predictive
signal beyond what the sales history alone provides.

To run the full-scale result on all 45 stores (matching the headline
figures in `comprehensive_report.txt`), call
`run_forecasting(df, test_weeks=8)` on the complete dataframe; this is
omitted from interactive execution here only for runtime, not because the
method differs.


## 9. Impact of External Factors

### Reasoning

CPI, unemployment, and fuel price are macroeconomic signals outside the
retailer's direct control, but they plausibly affect discretionary
spending. Three views are combined: a simple chain-level correlation, a
breakdown by store type (since premium and value stores may respond
differently to the same economic conditions), and a standardized linear
regression that estimates each factor's marginal association with sales
while holding store size and type constant — correlation alone could
otherwise be confounded by, for example, larger stores happening to sit
in lower-CPI regions.


In [ ]:
from external_factors import run_external_factors_analysis

ext_results = run_external_factors_analysis(df)

In [ ]:
ext_results['chain_correlation']

In [ ]:
ext_results['store_type_correlation']

In [ ]:
ext_results['regression_impact']

### Interpretation

Markdown spend shows the strongest positive association with sales among
the external factors, while unemployment and CPI show negative
associations — consistent with reduced discretionary spending under
economic strain. The standardized regression coefficients confirm these
directions hold even after controlling for store size and type, meaning
the relationships are not simply an artifact of larger stores operating
in different economic conditions. The effect sizes are modest relative to
store size itself, which remains the dominant driver of absolute sales
volume.


## 10. Personalization Strategies and Real-World Application

### Reasoning

The preceding sections produce segments, forecasts, and factor
relationships in isolation. This section synthesizes them into concrete,
segment-specific recommendations for inventory, pricing, and marketing —
generated directly from each segment's actual statistics rather than
fixed templates, so the recommendations would change if the underlying
data or clustering changed.


In [ ]:
from strategy import run_strategy_formulation

strategy_results = run_strategy_formulation(profile_clustered)

In [ ]:
strategy_results['inventory_strategy']

In [ ]:
strategy_results['pricing_strategy']

In [ ]:
strategy_results['marketing_strategy']

In [ ]:
strategy_results['real_world_challenges']

### Interpretation

Lower-volatility, higher-performing segments can safely run leaner safety
stock and lean on loyalty-based marketing rather than discounting.
Higher-volatility and lower-performing segments warrant larger safety
buffers and markdown-driven promotion to defend volume. These
recommendations should be treated as a starting hypothesis to validate
with a controlled pilot (e.g. adjusting safety stock in one segment for a
quarter and measuring stockout rate) before chain-wide rollout, given the
real-world challenges listed above — particularly store heterogeneity and
the absence of true transaction-level basket data.


## 11. Conclusion

This project delivered an end-to-end retail analytics pipeline covering
anomaly detection, time-based seasonality analysis, segmentation with
formal quality evaluation, market basket analysis, demand forecasting, and
external factor analysis, synthesized into segment-specific business
strategy.

**Key takeaways:**

- A consensus-based, multi-method approach to anomaly detection
  meaningfully reduces false positives compared to any single method.
- Store segmentation based on sales behavior substantially recovers the
  business's existing store Type structure, while also revealing
  finer-grained volatility differences within each type.
- Lag-based features dominate demand forecasting performance, but
  markdown and economic features add measurable incremental value.
- External economic factors have a real but secondary effect on sales
  relative to store size; their main practical use is in fine-tuning
  promotion timing rather than as a primary sales driver.
- The lack of transaction-level data is the most significant constraint
  on market basket analysis; results should be read as directional
  signals rather than confirmed co-purchase patterns.

**GitHub Repository:** _add link here before submission_

**Project Folder:** `retail_analytics_optimization` | **Environment:**
Anaconda, Python 3.10, `retail_ml`
